# Лабораторная работа №4 — вариант 7


In [1]:
%pip install pandas numpy

##  Загрузка и подготовка данных


In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

names = ["events_records.json", "events_lines.json"]
DATA = next((p for p in [Path("data"), Path(".")]
             if all((p / n).exists() for n in names)), Path("data"))
assert all((DATA / n).exists() for n in names), "Загрузите оба JSON в data или укажите нужную папку в DATA"
fields = ["event_id", "user_id", "event_time", "event_type", "page"]

def inspect(df, name):
    print(name, df.shape)
    display(df.head())
    df.info()
    display(df.dtypes)

raw = json.loads((DATA / names[0]).read_text(encoding="utf-8-sig"))
records = pd.DataFrame(raw)
if not set(fields).issubset(records.columns):
    assert len(records) == len(fields), "Неизвестная структура events_records.json"
    records = records.T
    records.columns = fields
    records = records.reset_index(drop=True)
inspect(records, names[0])

lines = pd.read_json(DATA / names[1], lines=True)
inspect(lines, names[1])
events = pd.concat([records[fields], lines[fields]], ignore_index=True)
assert not events[fields].isna().any().any(), "В исходных данных есть пропуски"
for column in ["event_id", "user_id"]:
    events[column] = pd.to_numeric(events[column], errors="raise").astype("int64")
events["event_time"] = pd.to_datetime(events["event_time"], errors="raise")
initial = len(events)
events = events.drop_duplicates().copy()
assert events["event_id"].is_unique, "Одинаковым event_id соответствуют разные записи"
print(f"Удалено одинаковых записей: {initial - len(events)}")
print(f"Событий: {len(events)}; пользователей: {events['user_id'].nunique()}")

events_records.json (2000, 5)


,event_id,user_id,event_time,event_type,page
0,1,1085,2024-01-30T20:37:32,page_view,/product/26
1,2,1006,2024-02-24T22:55:13,add_to_cart,/product/28
2,3,1024,2024-02-14T17:49:43,click,/product/27
3,4,1062,2024-02-16T11:21:32,click,/product/1
4,5,1012,2024-02-08T05:25:21,add_to_cart,/


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   event_id    2000 non-null   object
 1   user_id     2000 non-null   object
 2   event_time  2000 non-null   object
 3   event_type  2000 non-null   object
 4   page        2000 non-null   object
dtypes: object(5)
memory usage: 39.1+ KB


event_id      object
user_id       object
event_time    object
event_type    object
page          object
dtype: object

events_lines.json (2000, 5)


,event_id,user_id,event_time,event_type,page
0,2001,1029,2024-03-19 00:34:38,purchase,/product/20
1,2002,1056,2024-03-25 17:16:08,add_to_cart,/cart
2,2003,1081,2024-02-06 06:09:49,add_to_cart,/product/30
3,2004,1005,2024-03-20 04:35:23,purchase,/product/5
4,2005,1033,2024-02-21 09:51:53,click,/product/11


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   event_id    2000 non-null   int64         
 1   user_id     2000 non-null   int64         
 2   event_time  2000 non-null   datetime64[us]
 3   event_type  2000 non-null   str           
 4   page        2000 non-null   str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 62.6 KB


event_id               int64
user_id                int64
event_time    datetime64[us]
event_type               str
page                     str
dtype: object

Удалено одинаковых записей: 0
Событий: 4000; пользователей: 100


## 1. Интервалы и среднее для каждого пользователя
Для каждого пользователя посчитать средний интервал между событиями ( delta_since_prev ).


In [3]:
events = events.sort_values(["user_id", "event_time", "event_id"]).reset_index(drop=True)
events["delta_since_prev"] = events.groupby("user_id")["event_time"].diff()
assert events["delta_since_prev"].dropna().ge(pd.Timedelta(0)).all()
display(events[["user_id", "event_time", "delta_since_prev"]].head(10))

summary = events.groupby("user_id").agg(
    event_count=("event_id", "size"),
    interval_count=("delta_since_prev", "count"),
    mean_interval=("delta_since_prev", "mean"),
    first_event=("event_time", "min"),
    last_event=("event_time", "max")
)
summary["mean_hours"] = summary["mean_interval"].dt.total_seconds() / 3600
assert (summary["interval_count"] == summary["event_count"] - 1).all()
# Независимая формула: средний интервал = (последнее время - первое) / (n - 1).
valid = summary.loc[summary["interval_count"] > 0].copy()
expected = (valid["last_event"] - valid["first_event"]).dt.total_seconds() / valid["interval_count"]
assert np.allclose(valid["mean_interval"].dt.total_seconds(), expected, rtol=0, atol=1e-6)
display(summary[["event_count", "interval_count", "mean_interval", "mean_hours"]].assign(mean_hours=summary["mean_hours"].round(3)))

,user_id,event_time,delta_since_prev
0,1001,2024-01-01 06:25:31,NaT
1,1001,2024-01-05 07:12:56,4 days 00:47:25
2,1001,2024-01-08 15:40:51,3 days 08:27:55
3,1001,2024-01-09 06:39:26,0 days 14:58:35
4,1001,2024-01-15 00:00:38,5 days 17:21:12
5,1001,2024-01-20 00:29:26,5 days 00:28:48
6,1001,2024-01-24 08:41:28,4 days 08:12:02
7,1001,2024-01-24 11:54:49,0 days 03:13:21
8,1001,2024-01-25 17:54:45,1 days 05:59:56
9,1001,2024-01-26 17:20:09,0 days 23:25:24


,event_count,interval_count,mean_interval,mean_hours
user_id,,,,
1001,54,53,1 days 17:03:27.754716,41.058
1002,41,40,2 days 04:24:39.200000,52.411
1003,39,38,2 days 02:31:50.868421,50.531
1004,37,36,2 days 07:03:55.416666,55.065
1005,44,43,2 days 01:24:02.581395,49.401
...,...,...,...,...
1096,46,45,1 days 20:13:20.511111,44.222
1097,41,40,2 days 05:28:25.950000,53.474
1098,48,47,1 days 20:02:50.255319,44.047


## 3. Пользователи с минимальным и максимальным средним интервалом
В сравнение включаем пользователей с двумя и более событиями. Если крайнее значение одинаково у нескольких пользователей, показываем их всех. Сравниваем значения до округления.

In [4]:
assert not valid.empty, "Нет пользователей с двумя и более событиями"
minimum = valid.loc[valid["mean_interval"].eq(valid["mean_interval"].min())]
maximum = valid.loc[valid["mean_interval"].eq(valid["mean_interval"].max())]
print("Минимальный средний интервал:")
display(minimum[["event_count", "mean_interval", "mean_hours"]])
print("Максимальный средний интервал:")
display(maximum[["event_count", "mean_interval", "mean_hours"]])
print("Пользователей без вычислимого интервала:", len(summary) - len(valid))

Минимальный средний интервал:


,event_count,mean_interval,mean_hours
user_id,,,
1091,64,1 days 08:26:59.507936,32.449863


Максимальный средний интервал:


,event_count,mean_interval,mean_hours
user_id,,,
1082,24,3 days 18:34:49.608695,90.580447


Пользователей без вычислимого интервала: 0


## 4. Интерпретация активности
Меньший средний интервал означает, что зарегистрированные события пользователя происходят чаще. Это может указывать на более интенсивное взаимодействие с сайтом. Больший интервал означает более редкие события или длительные перерывы между посещениями.

Этот показатель сам по себе не доказывает заинтересованность или покупки: он зависит от количества событий, длины наблюдаемого периода, типов действий и перерывов между сессиями. Поэтому рядом со средним показано число событий. Для оценки действий внутри одного посещения потребовалось бы отдельно выделить сессии.

In [5]:
for label, frame in [("Чаще", minimum), ("Реже", maximum)]:
    for user_id, row in frame.iterrows():
        print(f"{label}: пользователь {user_id}, средний интервал {row['mean_hours']:.2f} ч, "
              f"событий {int(row['event_count'])}.")
summary.to_csv("user_intervals.csv", encoding="utf-8-sig")
print("Средние интервалы всех пользователей сохранены в user_intervals.csv")

Чаще: пользователь 1091, средний интервал 32.45 ч, событий 64.
Реже: пользователь 1082, средний интервал 90.58 ч, событий 24.
Средние интервалы всех пользователей сохранены в user_intervals.csv
